In [ ]:
# Testy analizy jasności w serii dwóch zdjęć - z lampą (lub podobnym oświetleniem) i bez
# Celem jest sprawdzenie, jak zmienia się jasność zdjęcia w zależności od oświetlenia
# W zależności, czy twarz jest autentyczna i znajduje się na prawdziwym tle, czy jest wyświetlona na ekranie
import cv2
import os
import matplotlib.pyplot as plt

In [ ]:
# Załadowanie przykładowych zdjęć
# W dzień i w nocy
original_daylight_nolight = cv2.imread("../../../Dane/Lighting/Original_daylight_nolight.png")
original_daylight_light = cv2.imread("../../../Dane/Lighting/Original_daylight_light.png")
spoof_daylight_nolight = cv2.imread("../../../Dane/Lighting/Spoof_daylight_nolight.png")
spoof_daylight_light = cv2.imread("../../../Dane/Lighting/Spoof_daylight_light.png")

original_nolight = cv2.imread("../../../Dane/Lighting/Original_nolight.png")
original_light = cv2.imread("../../../Dane/Lighting/Original_light.png")
spoof_nolight = cv2.imread("../../../Dane/Lighting/Spoof_nolight.png")
spoof_light = cv2.imread("../../../Dane/Lighting/Spoof_light.png")

# Wyświetlenie przykładowych zdjęć
plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.imshow(cv2.cvtColor(original_daylight_nolight, cv2.COLOR_BGR2RGB))
plt.title("Original daylight nolight")
plt.subplot(2, 2, 2)
plt.imshow(cv2.cvtColor(original_daylight_light, cv2.COLOR_BGR2RGB))
plt.title("Original daylight light")
plt.subplot(2, 2, 3)
plt.imshow(cv2.cvtColor(spoof_daylight_nolight, cv2.COLOR_BGR2RGB))
plt.title("Spoof daylight nolight")
plt.subplot(2, 2, 4)
plt.imshow(cv2.cvtColor(spoof_daylight_light, cv2.COLOR_BGR2RGB))
plt.title("Spoof daylight light")
plt.show()

In [ ]:
# Analiza zdjęć - do testów
# 1. Przetwarzanie wstępne obrazu
# 2. Porównanie jasności?
img_size = (256, 256)
def preprocess_image(image):
    image = cv2.resize(image, img_size)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return image

def brightness_analysis(image1, image2):
    return cv2.absdiff(image1, image2)

real_diff = brightness_analysis(preprocess_image(original_daylight_nolight), preprocess_image(original_daylight_light))
spoof_diff = brightness_analysis(preprocess_image(spoof_daylight_nolight), preprocess_image(spoof_daylight_light))

real_mean = real_diff.mean()
spoof_mean = spoof_diff.mean()
print("Daylight real mean diff: ", real_mean)
print("Daylight spoof mean diff: ", spoof_mean)

real_diff_night = brightness_analysis(preprocess_image(original_nolight), preprocess_image(original_light))
spoof_diff_night = brightness_analysis(preprocess_image(spoof_nolight), preprocess_image(spoof_light))

real_mean_night = real_diff_night.mean()
spoof_mean_night = spoof_diff_night.mean()

print("Night real mean diff: ", real_mean_night)
print("Night spoof mean diff: ", spoof_mean_night)

In [ ]:
# HSV
# 1. Przetwarzanie wstępne obrazu
# 2. Porównanie jasności?
def preprocess_image(image):
    image = cv2.resize(image, img_size)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    return image

def brightness_analysis(image1, image2):
    return cv2.absdiff(image1, image2)

original_daylight_hsv_nolight = preprocess_image(original_daylight_nolight)
original_daylight_hsv_light = preprocess_image(original_daylight_light)
spoof_daylight_hsv_nolight = preprocess_image(spoof_daylight_nolight)
spoof_daylight_hsv_light = preprocess_image(spoof_daylight_light)

real_diff_s = brightness_analysis(original_daylight_hsv_nolight[:,:,1], original_daylight_hsv_light[:,:,1])
real_diff_v = brightness_analysis(original_daylight_hsv_nolight[:,:,2], original_daylight_hsv_light[:,:,2])

print("Daylight real mean diff S: ", real_diff_s.mean())
print("Daylight real mean diff V: ", real_diff_v.mean())

spoof_diff_s = brightness_analysis(spoof_daylight_hsv_nolight[:,:,1], spoof_daylight_hsv_light[:,:,1])
spoof_diff_v = brightness_analysis(spoof_daylight_hsv_nolight[:,:,2], spoof_daylight_hsv_light[:,:,2])

print("Daylight spoof mean diff S: ", spoof_diff_s.mean())
print("Daylight spoof mean diff V: ", spoof_diff_v.mean())

In [ ]:
# To samo sprawdzenie dla zdjęć w nocy (czy są znaczące różnice np. w tle między dniem a nocą)
original_night_hsv_nolight = preprocess_image(original_nolight)
original_night_hsv_light = preprocess_image(original_light)
spoof_night_hsv_nolight = preprocess_image(spoof_nolight)
spoof_night_hsv_light = preprocess_image(spoof_light)

real_diff_s_night = brightness_analysis(original_night_hsv_nolight[:,:,1], original_night_hsv_light[:,:,1])
real_diff_v_night = brightness_analysis(original_night_hsv_nolight[:,:,2], original_night_hsv_light[:,:,2])

print("Night real mean diff S: ", real_diff_s_night.mean())
print("Night real mean diff V: ", real_diff_v_night.mean())

spoof_diff_s_night = brightness_analysis(spoof_night_hsv_nolight[:,:,1], spoof_night_hsv_light[:,:,1])
spoof_diff_v_night = brightness_analysis(spoof_night_hsv_nolight[:,:,2], spoof_night_hsv_light[:,:,2])

print("Night spoof mean diff S: ", spoof_diff_s_night.mean())
print("Night spoof mean diff V: ", spoof_diff_v_night.mean())

In [ ]:
#  Zdjęcie autentyczne w nocy ma większe różnice, niż nieautentyczne;
#  zdjęcie nieautentyczne ma podobne różnice w dzień i w nocy (V różni się trochę bardziej)
import cv2.data

def brightness_analysis(image1, image2):
    return cv2.absdiff(image1, image2)


def perform_hsv_analysis(image_nolight, image_light):
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    
    faces = face_cascade.detectMultiScale(image_nolight, 1.3, 6)
    x, y, w, h = faces[0]
    face_area = image_nolight[y:y+h, x:x+w]
    face_area_nolight_hsv = cv2.cvtColor(face_area, cv2.COLOR_BGR2HSV)
    
    faces = face_cascade.detectMultiScale(image_light, 1.3, 6)
    x, y, w, h = faces[0]
    face_area = image_light[y:y+h, x:x+w]
    face_area_light_hsv = cv2.cvtColor(face_area, cv2.COLOR_BGR2HSV)
    
    # Zmiana rozmiaru
    face_size = (64, 64)
    face_area_nolight_hsv = cv2.resize(face_area_nolight_hsv, face_size)
    face_area_light_hsv = cv2.resize(face_area_light_hsv, face_size)
    
    # Porównanie H, S i V
    diff_h = brightness_analysis(face_area_nolight_hsv[:,:,0], face_area_light_hsv[:,:,0])
    diff_s = brightness_analysis(face_area_nolight_hsv[:,:,1], face_area_light_hsv[:,:,1])
    diff_v = brightness_analysis(face_area_nolight_hsv[:,:,2], face_area_light_hsv[:,:,2])
    
    return diff_h.mean(), diff_s.mean(), diff_v.mean()

real_diff_h, real_diff_s, real_diff_v = perform_hsv_analysis(original_nolight, original_light)
print("Twarz autentyczna, noc, H: ", real_diff_h, " S: ", real_diff_s, " V: ", real_diff_v)

spoof_diff_h, spoof_diff_s, spoof_diff_v = perform_hsv_analysis(spoof_nolight, spoof_light)
print("Twarz podstawiona, noc, H: ", spoof_diff_h, " S: ", spoof_diff_s, " V: ", spoof_diff_v)

real_diff_h, real_diff_s, real_diff_v = perform_hsv_analysis(original_daylight_nolight, original_daylight_light)
print("Twarz autentyczna, dzień, H: ", real_diff_h, " S: ", real_diff_s, " V: ", real_diff_v)

spoof_diff_h, spoof_diff_s, spoof_diff_v = perform_hsv_analysis(spoof_daylight_nolight, spoof_daylight_light)
print("Twarz podstawiona, dzień, H: ", spoof_diff_h, " S: ", spoof_diff_s, " V: ", spoof_diff_v)

In [ ]:
# Powtórzenie powyższego, ale dla HLS zamiast HSV
def perform_hsl_analysis(image_nolight, image_light):
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    faces = face_cascade.detectMultiScale(image_nolight, 1.3, 6)
    x, y, w, h = faces[0]
    face_area = image_nolight[y:y+h, x:x+w]
    face_area_nolight_hsl = cv2.cvtColor(face_area, cv2.COLOR_BGR2HLS)

    faces = face_cascade.detectMultiScale(image_light, 1.3, 6)
    x, y, w, h = faces[0]
    face_area = image_light[y:y+h, x:x+w]
    face_area_light_hsl = cv2.cvtColor(face_area, cv2.COLOR_BGR2HLS)

    # Zmiana rozmiaru
    face_size = (64, 64)
    face_area_nolight_hsl = cv2.resize(face_area_nolight_hsl, face_size)
    face_area_light_hsl = cv2.resize(face_area_light_hsl, face_size)

    # Porównanie H, L i S
    diff_h = brightness_analysis(face_area_nolight_hsl[:,:,0], face_area_light_hsl[:,:,0])
    diff_l = brightness_analysis(face_area_nolight_hsl[:,:,1], face_area_light_hsl[:,:,1])
    diff_s = brightness_analysis(face_area_nolight_hsl[:,:,2], face_area_light_hsl[:,:,2])

    return diff_h.mean(), diff_l.mean(), diff_s.mean()

real_diff_h, real_diff_l, real_diff_s = perform_hsl_analysis(original_nolight, original_light)
print("Twarz autentyczna, noc, H: ", real_diff_h, " L: ", real_diff_l, " S: ", real_diff_s)

spoof_diff_h, spoof_diff_l, spoof_diff_s = perform_hsl_analysis(spoof_nolight, spoof_light)
print("Twarz podstawiona, noc, H: ", spoof_diff_h, " L: ", spoof_diff_l, " S: ", spoof_diff_s)

real_diff_h, real_diff_l, real_diff_s = perform_hsl_analysis(original_daylight_nolight, original_daylight_light)
print("Twarz autentyczna, dzień, H: ", real_diff_h, " L: ", real_diff_l, " S: ", real_diff_s)

spoof_diff_h, spoof_diff_l, spoof_diff_s = perform_hsl_analysis(spoof_daylight_nolight, spoof_daylight_light)
print("Twarz podstawiona, dzień, H: ", spoof_diff_h, " L: ", spoof_diff_l, " S: ", spoof_diff_s)

In [ ]:
# Powtórzenie tego wyżej, ale dla CIELUV (HCL?)
def perform_luv_analysis(image_nolight, image_light):
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    faces = face_cascade.detectMultiScale(image_nolight, 1.1, 5)
    x, y, w, h = faces[0]
    face_area = image_nolight[y:y+h, x:x+w]
    face_area_nolight_luv = cv2.cvtColor(face_area, cv2.COLOR_BGR2Luv)

    faces = face_cascade.detectMultiScale(image_light, 1.1, 5)
    x, y, w, h = faces[0]
    face_area = image_light[y:y+h, x:x+w]
    face_area_light_luv = cv2.cvtColor(face_area, cv2.COLOR_BGR2Luv)

    # Zmiana rozmiaru
    face_size = (64, 64)
    face_area_nolight_luv = cv2.resize(face_area_nolight_luv, face_size)
    face_area_light_luv = cv2.resize(face_area_light_luv, face_size)

    # Porównanie L, U i V
    diff_l = brightness_analysis(face_area_nolight_luv[:,:,0], face_area_light_luv[:,:,0])
    diff_u = brightness_analysis(face_area_nolight_luv[:,:,1], face_area_light_luv[:,:,1])
    diff_v = brightness_analysis(face_area_nolight_luv[:,:,2], face_area_light_luv[:,:,2])

    return diff_l.mean(), diff_u.mean(), diff_v.mean()

real_diff_l, real_diff_u, real_diff_v = perform_luv_analysis(original_nolight, original_light)
print("Twarz autentyczna, noc, L: ", real_diff_l, " U: ", real_diff_u, " V: ", real_diff_v)

spoof_diff_l, spoof_diff_u, spoof_diff_v = perform_luv_analysis(spoof_nolight, spoof_light)
print("Twarz podstawiona, noc, L: ", spoof_diff_l, " U: ", spoof_diff_u, " V: ", spoof_diff_v)

real_diff_l, real_diff_u, real_diff_v = perform_luv_analysis(original_daylight_nolight, original_daylight_light)
print("Twarz autentyczna, dzień, L: ", real_diff_l, " U: ", real_diff_u, " V: ", real_diff_v)

spoof_diff_l, spoof_diff_u, spoof_diff_v = perform_luv_analysis(spoof_daylight_nolight, spoof_daylight_light)
print("Twarz podstawiona, dzień, L: ", spoof_diff_l, " U: ", spoof_diff_u, " V: ", spoof_diff_v)

In [ ]:
# Saturation jest identyczne dla twarzy autentycznej i podstawionej za dnia
# Przetestować powyższą funkcję dla większej ilości zdjęć
# Sprawdzać zdjęcia za dnia: w szczególności podstawione
# Chyba nie można sprawdzać po wartościach H, bo to jest tylko kolor?
original_light_2 = cv2.imread("../../../Dane/Lighting/original2_light.jpg")
original_nolight_2 = cv2.imread("../../../Dane/Lighting/original2_nolight.jpg")

real_diff_2_l, real_diff_2_u, real_diff_2_v = perform_luv_analysis(original_nolight_2, original_light_2)
print("Twarz autentyczna 2, L: ", real_diff_2_l, " U: ", real_diff_2_u, " V: ", real_diff_2_v)

spoof_light_2 = cv2.imread("../../../Dane/Lighting/spoof2_light.png")
spoof_nolight_2 = cv2.imread("../../../Dane/Lighting/spoof2_nolight.png")

spoof_diff_2_l, spoof_diff_2_u, spoof_diff_2_v = perform_luv_analysis(spoof_nolight_2, spoof_light_2)
print("Twarz podstawiona 2, L: ", spoof_diff_2_l, " U: ", spoof_diff_2_u, " V: ", spoof_diff_2_v)

spoof_light_3 = cv2.imread("../../../Dane/Lighting/spoof3_light.png")
spoof_nolight_3 = cv2.imread("../../../Dane/Lighting/spoof3_nolight.png")

spoof_diff_3_l, spoof_diff_3_u, spoof_diff_3_v = perform_luv_analysis(spoof_nolight_3, spoof_light_3)
print("Twarz podstawiona 3, L: ", spoof_diff_3_l, " U: ", spoof_diff_3_u, " V: ", spoof_diff_3_v)

spoof_light_4 = cv2.imread("../../../Dane/Lighting/spoof4_light.png")
spoof_nolight_4 = cv2.imread("../../../Dane/Lighting/spoof4_nolight.png")

spoof_diff_4_l, spoof_diff_4_u, spoof_diff_4_v = perform_luv_analysis(spoof_nolight_4, spoof_light_4)
print("Twarz podstawiona 4, L: ", spoof_diff_4_l, " U: ", spoof_diff_4_u, " V: ", spoof_diff_4_v)

In [ ]:
# Wnioski:
# Jest spora różnica między autentycznym a nieautentycznym (dla samej twarzy) -> dałoby się to porównywać
# Jeżeli tło na zdjęciu jest bardzo jasne (np. białe), to różnica jest znikoma

In [ ]:
# Spróbować wyciąć pasek dłuższy albo szerszy niż wykryta twarz i sprawdzić dla nich
def perform_luv_analysis(image_nolight, image_light):
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    faces = face_cascade.detectMultiScale(image_nolight, 1.1, 5)
    x, y, w, h = faces[0]
    face_area = image_nolight[:, x:x+w]
    face_area_nolight_luv = cv2.cvtColor(face_area, cv2.COLOR_BGR2Luv)

    faces = face_cascade.detectMultiScale(image_light, 1.1, 5)
    x, y, w, h = faces[0]
    face_area = image_light[:, x:x+w]
    face_area_light_luv = cv2.cvtColor(face_area, cv2.COLOR_BGR2Luv)

    # Zmiana rozmiaru
    face_size = (256, 64)
    face_area_nolight_luv = cv2.resize(face_area_nolight_luv, face_size)
    face_area_light_luv = cv2.resize(face_area_light_luv, face_size)

    # Porównanie L, U i V
    diff_l = brightness_analysis(face_area_nolight_luv[:,:,0], face_area_light_luv[:,:,0])
    diff_u = brightness_analysis(face_area_nolight_luv[:,:,1], face_area_light_luv[:,:,1])
    diff_v = brightness_analysis(face_area_nolight_luv[:,:,2], face_area_light_luv[:,:,2])

    return diff_l.mean(), diff_u.mean(), diff_v.mean()

real_diff_l, real_diff_u, real_diff_v = perform_luv_analysis(original_daylight_light, original_daylight_nolight)
print("Twarz autentyczna 1, L: ", real_diff_l, " U: ", real_diff_u, " V: ", real_diff_v)

real_diff_2_l, real_diff_2_u, real_diff_2_v = perform_luv_analysis(original_light_2, original_nolight_2)
print("Twarz autentyczna 2, L: ", real_diff_2_l, " U: ", real_diff_2_u, " V: ", real_diff_2_v)

spoof_diff_l, spoof_diff_u, spoof_diff_v = perform_luv_analysis(spoof_daylight_light, spoof_daylight_nolight)
print("Twarz podstawiona 1, L: ", spoof_diff_l, " U: ", spoof_diff_u, " V: ", spoof_diff_v)

spoof_diff_2_l, spoof_diff_2_u, spoof_diff_2_v = perform_luv_analysis(spoof_light_2, spoof_nolight_2)
print("Twarz podstawiona 2, L: ", spoof_diff_2_l, " U: ", spoof_diff_2_u, " V: ", spoof_diff_2_v)

spoof_diff_3_l, spoof_diff_3_u, spoof_diff_3_v = perform_luv_analysis(spoof_light_3, spoof_nolight_3)
print("Twarz podstawiona 3, L: ", spoof_diff_3_l, " U: ", spoof_diff_3_u, " V: ", spoof_diff_3_v)

spoof_diff_4_l, spoof_diff_4_u, spoof_diff_4_v = perform_luv_analysis(spoof_light_4, spoof_nolight_4)
print("Twarz podstawiona 4, L: ", spoof_diff_4_l, " U: ", spoof_diff_4_u, " V: ", spoof_diff_4_v)

In [ ]:
# A teraz całe zdjęcia
def perform_luv_analysis(image_nolight, image_light):
    # Zmiana rozmiaru
    image_size = (256, 256)
    face_area_nolight_luv = cv2.resize(image_nolight, image_size)
    face_area_light_luv = cv2.resize(image_light, image_size)

    # Porównanie L, U i V
    diff_l = brightness_analysis(face_area_nolight_luv[:,:,0], face_area_light_luv[:,:,0])
    diff_u = brightness_analysis(face_area_nolight_luv[:,:,1], face_area_light_luv[:,:,1])
    diff_v = brightness_analysis(face_area_nolight_luv[:,:,2], face_area_light_luv[:,:,2])

    return diff_l.mean(), diff_u.mean(), diff_v.mean()

real_diff_l, real_diff_u, real_diff_v = perform_luv_analysis(original_daylight_light, original_daylight_nolight)
print("Twarz autentyczna 1, L: ", real_diff_l, " U: ", real_diff_u, " V: ", real_diff_v)

real_diff_2_l, real_diff_2_u, real_diff_2_v = perform_luv_analysis(original_light_2, original_nolight_2)
print("Twarz autentyczna 2, L: ", real_diff_2_l, " U: ", real_diff_2_u, " V: ", real_diff_2_v)

spoof_diff_l, spoof_diff_u, spoof_diff_v = perform_luv_analysis(spoof_daylight_light, spoof_daylight_nolight)
print("Twarz podstawiona 1, L: ", spoof_diff_l, " U: ", spoof_diff_u, " V: ", spoof_diff_v)

spoof_diff_2_l, spoof_diff_2_u, spoof_diff_2_v = perform_luv_analysis(spoof_light_2, spoof_nolight_2)
print("Twarz podstawiona 2, L: ", spoof_diff_2_l, " U: ", spoof_diff_2_u, " V: ", spoof_diff_2_v)

spoof_diff_3_l, spoof_diff_3_u, spoof_diff_3_v = perform_luv_analysis(spoof_light_3, spoof_nolight_3)
print("Twarz podstawiona 3, L: ", spoof_diff_3_l, " U: ", spoof_diff_3_u, " V: ", spoof_diff_3_v)

spoof_diff_4_l, spoof_diff_4_u, spoof_diff_4_v = perform_luv_analysis(spoof_light_4, spoof_nolight_4)
print("Twarz podstawiona 4, L: ", spoof_diff_4_l, " U: ", spoof_diff_4_u, " V: ", spoof_diff_4_v)

In [ ]:
# Wnioski:
# Chyba tego typu metoda jest lepsza (tylko przetestować rozmiary i szerokość/wysokość - pasek czy całe zdjęcie?)
# Dla twarzy autentycznych różnica jest nieznaczna
# Dla twarzy podstawionych jest ogromna
# Jednak dla jednolitego tła (np. białego), różnica jest niewielka, nie wykrywamy poprawnie zmiany oświetlenia

In [ ]:
# Do pracy: wyświetlenie twarzy w RGB i LUV
normal_img = cv2.resize(cv2.cvtColor(spoof_daylight_light, cv2.COLOR_BGR2GRAY), (256, 256))
luv_img = cv2.resize(cv2.cvtColor(spoof_daylight_light, cv2.COLOR_BGR2Luv), (256, 256))

plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(normal_img, cmap="gray")
plt.title("RGB")
plt.axis('off')

luv_img_normalized = luv_img[:,:,0]

plt.subplot(1, 2, 2)
plt.imshow(luv_img_normalized, cmap='gray')
plt.title("LUV")
plt.axis('off')
plt.show()

In [ ]:
print(luv_img_normalized)

In [ ]:
print(normal_img)

In [ ]:
# Sprawdzić ile się pokrywa
count = 0
for i in range(256):
    for j in range(256):
        if normal_img[i][j] == luv_img_normalized[i][j]:
            count += 1
print(count)

In [ ]:
# Wniosek: są różnice między takim standardowym grayscale a LUV
# Na 65536 pokrywa się tylko 1439